# Online Retail Customer & Sales Analytics

## Import Libraries

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

## Loading Dataset

In [8]:
df = pd.read_csv(r"D:\Datasets\Online Retail data.csv")
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


## Data Understanding

In [9]:
print("Shape:", df.shape)
df.info()
df.describe()

Shape: (541909, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


## Checking Missing Values

In [13]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

## Data Cleaning

In [29]:
# Remove missing CustomerID
df = df.dropna(subset=['CustomerID'])

# Remove negative/zero quantity (returns or errors)
df = df[df['Quantity'] > 0]

# Remove negative price
df = df[df['UnitPrice'] > 0]

# Convert date
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Create TotalPrice
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

## saving cleaned data
df.to_csv("cleaned_retail_data.csv", index=False)

## Create RFM Features

In [14]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'InvoiceNo': 'nunique',                                  # Frequency
    'TotalPrice': 'sum'}).reset_index()                      # Monetary

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

## Feature Engineering

In [15]:
# Average Order Value
rfm['AOV'] = rfm['Monetary'] / rfm['Frequency']

# Customer Lifetime
first_purchase = df.groupby('CustomerID')['InvoiceDate'].min()
last_purchase = df.groupby('CustomerID')['InvoiceDate'].max()

rfm['Customer_Lifetime'] = (last_purchase - first_purchase).dt.days.values

## Creating Target (Churn)

In [16]:
# Churn: no purchase in last 90 days
rfm['Churn'] = rfm['Recency'].apply(lambda x: 1 if x > 90 else 0)

# Check distribution
print(rfm['Churn'].value_counts())

Churn
0    2889
1    1449
Name: count, dtype: int64


## Prepare Data

In [17]:
X = rfm[['Recency', 'Frequency', 'Monetary', 'AOV', 'Customer_Lifetime']]
y = rfm['Churn']

X_train, X_test, y_train, y_test = train_test_split (X, y, test_size=0.2, random_state=42)

## Logistic Regression

In [19]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:, 1]

print("=== Logistic Regression ===")
print(classification_report(y_test, lr_pred))
print("ROC-AUC:", roc_auc_score(y_test, lr_prob))

=== Logistic Regression ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       561
           1       1.00      1.00      1.00       307

    accuracy                           1.00       868
   macro avg       1.00      1.00      1.00       868
weighted avg       1.00      1.00      1.00       868

ROC-AUC: 1.0


## Random Forest

In [21]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("\n=== Random Forest ===")
print(classification_report(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_prob))


=== Random Forest ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       561
           1       1.00      1.00      1.00       307

    accuracy                           1.00       868
   macro avg       1.00      1.00      1.00       868
weighted avg       1.00      1.00      1.00       868

ROC-AUC: 1.0


## Feature Importance

In [22]:
importance = pd.Series(rf.feature_importances_, index=X.columns)
print("\nFeature Importance:")
print(importance.sort_values(ascending=False))


Feature Importance:
Recency              0.873535
Customer_Lifetime    0.067541
Frequency            0.029078
Monetary             0.023865
AOV                  0.005982
dtype: float64


## Churn Probability

In [23]:
rfm['Churn_Probability'] = rf.predict_proba(X)[:, 1]

# High-risk customers
high_risk = rfm[rfm['Churn_Probability'] > 0.7]

print("\nHigh Risk Customers:")
print(high_risk.head())


High Risk Customers:
   CustomerID  Recency  Frequency  Monetary      AOV  Customer_Lifetime  \
0     12346.0      326          1   77183.6  77183.6                  0   
4     12350.0      310          1     334.4    334.4                  0   
6     12353.0      204          1      89.0     89.0                  0   
7     12354.0      232          1    1079.4   1079.4                  0   
8     12355.0      214          1     459.4    459.4                  0   

   Churn  Churn_Probability  
0      1               0.99  
4      1               1.00  
6      1               1.00  
7      1               1.00  
8      1               1.00  


## Saving output

In [26]:
rfm.to_csv("customer_churn_predictions.csv", index=False)

## Conclusion